In [ ]:
!pip install -q captum shap lime

# Cuadernillo Experimental XAI: Interpretabilidad Mecanicista en Detección de Redundancia Textual en Español

**Proyecto de Tesis:** Cuantificación y Detección de Redundancia Semántica en Corpus Periodístico y Benchmarks en Español  
**Modelo Ganador Evaluado:** Sentence-BERT Clásico con Mean-Pooling (`hiiamsid/sentence_similarity_spanish_es`)  
**Autor:** Senior NLP Researcher & Lead Data Scientist  
**Entorno de Ejecución:** CPU Optimizado con Gestión Estricta de Memoria RAM y Recolección de Basura (`gc.collect`)

---

## Objetivos del Cuadernillo
1. Evaluar la interpretabilidad a nivel de token, palabra y oración de la arquitectura siamesa Sentence-BERT ganadora de la Fase 1.
2. Implementar y comparar cinco métodos de IA Explicable (XAI):
   - **Saliency / Gradiente Directo**
   - **Input × Gradient**
   - **Fast Integrated Gradients (Fast-IG: 8-10 pasos)**
   - **Mapas de Auto-Atención y Similitud Cruzada Inter-Oracional**
   - **LIME-Light (25 perturbaciones locales)**
   - **KernelSHAP-Light (25 coaliciones de Shapley)**
3. Ejecutar pruebas cuantitativas de fidelidad (*Faithfulness*): **Comprensividad (Erasure)** y **Suficiencia**.
4. Construir curvas de degradación por perturbación (*Feature Ablation Curves*): **MoRF** (*Most Relevant First*) vs. **LoRF** (*Least Relevant First*).
5. Realizar la prueba de sanidad metodológica (*Sanity Check* de Adebayo et al., 2018) mediante **Aleatorización en Cascada de Parámetros** de las capas del Transformer.
6. Guardar automáticamente todas las visualizaciones en alta resolución en `C:/Users/Usuario/Documents/tesis/reportes/imagenes/`.

In [ ]:
import os
import gc
import time
import math
import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import Ridge, LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel

# Configuración de rutas y estilos
BASE_DIR = "C:/Users/Usuario/Documents/tesis"
IMAGES_DIR = os.path.join(BASE_DIR, "reportes", "imagenes")
REPORT_DIR = os.path.join(BASE_DIR, "reportes")
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})

np.random.seed(42)
torch.manual_seed(42)

print("Entorno configurado correctamente. Dispositivo:", "cuda" if torch.cuda.is_available() else "cpu")

## 1. Suite de Evaluación Balanceada de 10 Pares Curados

Para evaluar con rigor mecanicista el comportamiento del modelo, se seleccionó una suite de **10 pares de oraciones altamente representativos** en español:
- **5 Pares Redundantes:** Oraciones con alta similitud conceptual, paráfrasis o equivalencia semántica directa.
- **5 Pares No Redundantes (Ortogonales):** Oraciones disímiles, pertenecientes a dominios no relacionados o con polaridad informativa opuesta.

In [ ]:
PAIRS = [
    # 5 Pares Redundantes
    {
        "id": 1,
        "type": "Redundante",
        "text_a": "El vehículo aceleró rápidamente en la autopista.",
        "text_b": "El auto aumentó su velocidad de forma veloz en la carretera."
    },
    {
        "id": 2,
        "type": "Redundante",
        "text_a": "Las fuertes lluvias causaron graves inundaciones en la ciudad.",
        "text_b": "El temporal de agua provocó que la urbe metropolitana se anegara."
    },
    {
        "id": 3,
        "type": "Redundante",
        "text_a": "El presidente anunció nuevas medidas económicas para el país.",
        "text_b": "El mandatario comunicó recientes políticas financieras a nivel nacional."
    },
    {
        "id": 4,
        "type": "Redundante",
        "text_a": "El equipo local ganó el campeonato tras un partido difícil.",
        "text_b": "El conjunto de casa se coronó campeón del torneo luego de un encuentro complejo."
    },
    {
        "id": 5,
        "type": "Redundante",
        "text_a": "Este descubrimiento científico cambiará el futuro de la medicina.",
        "text_b": "Este hallazgo de la ciencia transformará el área médica en los próximos años."
    },
    # 5 Pares No Redundantes (Ortogonales)
    {
        "id": 6,
        "type": "No Redundante",
        "text_a": "El perro ladra fuertemente en el jardín trasero.",
        "text_b": "El gato duerme plácidamente en el sofá de la sala."
    },
    {
        "id": 7,
        "type": "No Redundante",
        "text_a": "La bolsa de valores cayó drásticamente el día de hoy.",
        "text_b": "El nuevo restaurante de comida italiana abrió sus puertas."
    },
    {
        "id": 8,
        "type": "No Redundante",
        "text_a": "La inteligencia artificial avanza a pasos agigantados.",
        "text_b": "La receta de la abuela lleva mucha canela y azúcar."
    },
    {
        "id": 9,
        "type": "No Redundante",
        "text_a": "El pronóstico indica que el clima estará soleado mañana.",
        "text_b": "Las elecciones presidenciales de este año serán muy reñidas."
    },
    {
        "id": 10,
        "type": "No Redundante",
        "text_a": "El vuelo internacional fue cancelado por la tormenta de nieve.",
        "text_b": "El libro de ciencia ficción se convirtió en un best-seller mundial."
    }
]

print(f"Cargados exitosamente {len(PAIRS)} pares para experimentación XAI.")

## 2. Carga Única del Modelo en RAM y Función de Similitud Coseno

Se instancia una única vez el modelo Sentence-BERT pre-entrenado en español (`hiiamsid/sentence_similarity_spanish_es`) y se define la función de agregación *Mean-Pooling* con máscara de atención:

$$\mathbf{u} = \frac{\sum_{i=1}^{L} \mathbf{h}_i \cdot m_i}{\sum_{i=1}^{L} m_i}, \quad \mathbf{v} = \frac{\mathbf{u}}{\|\mathbf{u}\|_2}$$
$$\text{Sim}(T_A, T_B) = \cos(\mathbf{v}_A, \mathbf{v}_B) = \mathbf{v}_A^\top \mathbf{v}_B$$

In [ ]:
MODEL_NAME = "hiiamsid/sentence_similarity_spanish_es"
print(f"Cargando modelo y tokenizador: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval()

def mean_pool(h, mask):
    m = mask.unsqueeze(-1).expand(h.size()).float()
    return (h * m).sum(dim=1) / torch.clamp(m.sum(dim=1), min=1e-9)

def compute_similarity(text_a, text_b, curr_model=model):
    tok_a = tokenizer(text_a, return_tensors='pt', padding=True, truncation=True)
    tok_b = tokenizer(text_b, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        out_a = curr_model(**tok_a)[0]
        out_b = curr_model(**tok_b)[0]
        vec_a = F.normalize(mean_pool(out_a, tok_a['attention_mask']), p=2, dim=1)
        vec_b = F.normalize(mean_pool(out_b, tok_b['attention_mask']), p=2, dim=1)
        return float((vec_a * vec_b).sum().item())

print("Modelo instanciado y preparado en memoria.")

## 3. Métodos Basados en Gradiente: Saliency e Input × Gradient

En una red siamesa, calculamos la derivada de la puntuación de similitud escalar $\text{Sim}(E_A, E_B)$ con respecto a los embeddings de entrada $E_A$ y $E_B$:

- **Gradiente / Saliency:**
  $$S_i^{\text{sal}} = \left\| \frac{\partial \text{Sim}}{\partial E_i} \right\|_2$$
- **Input × Gradient:**
  $$S_i^{\text{IxG}} = \sum_{k=1}^{d} E_{i, k} \cdot \frac{\partial \text{Sim}}{\partial E_{i, k}}$$

In [ ]:
def explain_gradients(text_a, text_b, curr_model=model):
    t0 = time.perf_counter()
    tok_a = tokenizer(text_a, return_tensors='pt')
    tok_b = tokenizer(text_b, return_tensors='pt')
    
    emb_layer = curr_model.get_input_embeddings()
    embeds_a = emb_layer(tok_a['input_ids']).clone().detach().requires_grad_(True)
    embeds_b = emb_layer(tok_b['input_ids']).clone().detach().requires_grad_(True)
    
    out_a = curr_model(inputs_embeds=embeds_a, attention_mask=tok_a['attention_mask'])[0]
    out_b = curr_model(inputs_embeds=embeds_b, attention_mask=tok_b['attention_mask'])[0]
    
    vec_a = F.normalize(mean_pool(out_a, tok_a['attention_mask']), p=2, dim=1)
    vec_b = F.normalize(mean_pool(out_b, tok_b['attention_mask']), p=2, dim=1)
    
    sim = (vec_a * vec_b).sum()
    sim.backward()
    
    tokens_a = tokenizer.convert_ids_to_tokens(tok_a['input_ids'][0])
    tokens_b = tokenizer.convert_ids_to_tokens(tok_b['input_ids'][0])
    
    grad_a = embeds_a.grad[0]
    grad_b = embeds_b.grad[0]
    
    saliency_a = grad_a.norm(dim=-1).detach().cpu().numpy()
    saliency_b = grad_b.norm(dim=-1).detach().cpu().numpy()
    
    ixg_a = (embeds_a.detach()[0] * grad_a).sum(dim=-1).cpu().numpy()
    ixg_b = (embeds_b.detach()[0] * grad_b).sum(dim=-1).cpu().numpy()
    
    latency = time.perf_counter() - t0
    return {
        "tokens_a": tokens_a,
        "tokens_b": tokens_b,
        "saliency_a": saliency_a,
        "saliency_b": saliency_b,
        "ixg_a": ixg_a,
        "ixg_b": ixg_b,
        "similarity": float(sim.item()),
        "latency_sec": latency
    }

## 4. Fast Integrated Gradients (Fast-IG: 8-10 Pasos)

Aproxima la integral de camino sobre una interpolación lineal entre un vector base nulo $E^0 = \mathbf{0}$ y el embedding real $E$:

$$\text{IG}_i(E) = (E_i - E_i^0) \odot \frac{1}{M} \sum_{m=1}^{M} \left. \frac{\partial \text{Sim}}{\partial E_i} \right|_{E = E^0 + \frac{m}{M}(E - E^0)}$$

In [ ]:
def explain_fast_ig(text_a, text_b, steps=8, curr_model=model):
    t0 = time.perf_counter()
    tok_a = tokenizer(text_a, return_tensors='pt')
    tok_b = tokenizer(text_b, return_tensors='pt')
    
    emb_layer = curr_model.get_input_embeddings()
    embeds_a = emb_layer(tok_a['input_ids']).clone().detach()
    embeds_b = emb_layer(tok_b['input_ids']).clone().detach()
    
    base_a = torch.zeros_like(embeds_a)
    base_b = torch.zeros_like(embeds_b)
    
    # 1. Gradients for Text A (with fixed B representation)
    with torch.no_grad():
        out_b_fixed = curr_model(inputs_embeds=embeds_b, attention_mask=tok_b['attention_mask'])[0]
        vec_b_fixed = F.normalize(mean_pool(out_b_fixed, tok_b['attention_mask']), p=2, dim=1)
        
    accum_grad_a = torch.zeros_like(embeds_a)
    for step in range(1, steps + 1):
        alpha = step / steps
        interp_a = (base_a + alpha * (embeds_a - base_a)).clone().detach().requires_grad_(True)
        out_a = curr_model(inputs_embeds=interp_a, attention_mask=tok_a['attention_mask'])[0]
        vec_a = F.normalize(mean_pool(out_a, tok_a['attention_mask']), p=2, dim=1)
        sim = (vec_a * vec_b_fixed).sum()
        sim.backward()
        accum_grad_a += interp_a.grad
        
    avg_grad_a = accum_grad_a / steps
    ig_a = ((embeds_a - base_a) * avg_grad_a).sum(dim=-1)[0].detach().cpu().numpy()
    
    # 2. Gradients for Text B (with fixed A representation)
    with torch.no_grad():
        out_a_fixed = curr_model(inputs_embeds=embeds_a, attention_mask=tok_a['attention_mask'])[0]
        vec_a_fixed = F.normalize(mean_pool(out_a_fixed, tok_a['attention_mask']), p=2, dim=1)
        
    accum_grad_b = torch.zeros_like(embeds_b)
    for step in range(1, steps + 1):
        alpha = step / steps
        interp_b = (base_b + alpha * (embeds_b - base_b)).clone().detach().requires_grad_(True)
        out_b = curr_model(inputs_embeds=interp_b, attention_mask=tok_b['attention_mask'])[0]
        vec_b = F.normalize(mean_pool(out_b, tok_b['attention_mask']), p=2, dim=1)
        sim = (vec_a_fixed * vec_b).sum()
        sim.backward()
        accum_grad_b += interp_b.grad
        
    avg_grad_b = accum_grad_b / steps
    ig_b = ((embeds_b - base_b) * avg_grad_b).sum(dim=-1)[0].detach().cpu().numpy()
    
    tokens_a = tokenizer.convert_ids_to_tokens(tok_a['input_ids'][0])
    tokens_b = tokenizer.convert_ids_to_tokens(tok_b['input_ids'][0])
    
    latency = time.perf_counter() - t0
    return {
        "tokens_a": tokens_a,
        "tokens_b": tokens_b,
        "ig_a": ig_a,
        "ig_b": ig_b,
        "latency_sec": latency
    }

## 5. Mapas de Auto-Atención y Similitud Cruzada Inter-Oracional

Se extraen los pesos de auto-atención multi-cabeza de la capa 12 y se computa la matriz de alineación semántica cruzada de tokens entre $T_A$ y $T_B$:

$$M_{\text{cross}}(i, j) = \frac{\mathbf{h}_{A, i}^\top \mathbf{h}_{B, j}}{\|\mathbf{h}_{A, i}\|_2 \|\mathbf{h}_{B, j}\|_2}$$

In [ ]:
def explain_attention(text_a, text_b, curr_model=model):
    t0 = time.perf_counter()
    tok_a = tokenizer(text_a, return_tensors='pt')
    tok_b = tokenizer(text_b, return_tensors='pt')
    
    with torch.no_grad():
        out_a = curr_model(**tok_a)
        out_b = curr_model(**tok_b)
        
    attn_a = out_a.attentions[-1][0].mean(dim=0).cpu().numpy()
    attn_b = out_b.attentions[-1][0].mean(dim=0).cpu().numpy()
    
    attn_score_a = attn_a.sum(axis=0) / attn_a.shape[0]
    attn_score_b = attn_b.sum(axis=0) / attn_b.shape[0]
    
    h_a = out_a[0][0]
    h_b = out_b[0][0]
    h_a_norm = F.normalize(h_a, p=2, dim=-1)
    h_b_norm = F.normalize(h_b, p=2, dim=-1)
    cross_sim = torch.mm(h_a_norm, h_b_norm.t()).cpu().numpy()
    
    tokens_a = tokenizer.convert_ids_to_tokens(tok_a['input_ids'][0])
    tokens_b = tokenizer.convert_ids_to_tokens(tok_b['input_ids'][0])
    
    latency = time.perf_counter() - t0
    return {
        "tokens_a": tokens_a,
        "tokens_b": tokens_b,
        "attn_a": attn_a,
        "attn_b": attn_b,
        "attn_score_a": attn_score_a,
        "attn_score_b": attn_score_b,
        "cross_sim": cross_sim,
        "latency_sec": latency
    }

## 6. Métodos de Perturbación: LIME-Light y KernelSHAP-Light

- **LIME-Light (25 perturbaciones):** Ajusta una regresión lineal Ridge ponderada por proximidad exponencial sobre máscaras binarias de palabras.
- **KernelSHAP-Light (25 coaliciones):** Ajusta una regresión ponderada por el núcleo de Shapley para estimar los valores marginales de Shapley sobre la métrica de similitud coseno.

In [ ]:
def explain_lime_light(text_a, text_b, n_samples=25, curr_model=model):
    t0 = time.perf_counter()
    def get_lime_scores(target_text, reference_text):
        words = target_text.split()
        n = len(words)
        np.random.seed(42)
        masks = np.random.binomial(1, 0.7, size=(n_samples, n))
        masks[0] = 1
        
        sims = []
        weights = []
        for mask in masks:
            sub_words = [words[i] for i in range(n) if mask[i] == 1]
            perturbed = ' '.join(sub_words) if sub_words else '[PAD]'
            s = compute_similarity(perturbed, reference_text, curr_model)
            sims.append(s)
            dist = np.sum(1 - mask) / max(n, 1)
            weights.append(np.exp(- (dist ** 2) / 0.25))
            
        reg = Ridge(alpha=1.0)
        reg.fit(masks, sims, sample_weight=weights)
        return words, reg.coef_
        
    words_a, coefs_a = get_lime_scores(text_a, text_b)
    words_b, coefs_b = get_lime_scores(text_b, text_a)
    latency = time.perf_counter() - t0
    return {
        "words_a": words_a,
        "words_b": words_b,
        "lime_a": coefs_a,
        "lime_b": coefs_b,
        "latency_sec": latency
    }

def explain_shap_light(text_a, text_b, n_samples=25, curr_model=model):
    t0 = time.perf_counter()
    def get_shap_scores(target_text, reference_text):
        words = target_text.split()
        n = len(words)
        np.random.seed(42)
        masks = np.random.binomial(1, 0.5, size=(n_samples, n))
        masks[0] = 1
        masks[1] = 0
        
        sims = []
        weights = []
        for mask in masks:
            k = int(np.sum(mask))
            if k == 0 or k == n:
                weight = 1000.0
            else:
                weight = (n - 1) / (math.comb(n, k) * k * (n - k))
            sub_words = [words[i] for i in range(n) if mask[i] == 1]
            perturbed = ' '.join(sub_words) if sub_words else '[PAD]'
            sims.append(compute_similarity(perturbed, reference_text, curr_model))
            weights.append(weight)
            
        reg = LinearRegression()
        reg.fit(masks, sims, sample_weight=weights)
        return words, reg.coef_
        
    words_a, coefs_a = get_shap_scores(text_a, text_b)
    words_b, coefs_b = get_shap_scores(text_b, text_a)
    latency = time.perf_counter() - t0
    return {
        "words_a": words_a,
        "words_b": words_b,
        "shap_a": coefs_a,
        "shap_b": coefs_b,
        "latency_sec": latency
    }

## 7. Métricas Cuantitativas de Fidelidad, Curvas MoRF/LoRF y Prueba de Sanidad

- **Comprensividad (Erasure):** Caída en la similitud coseno tras eliminar el top 20% de tokens explicados.
- **Suficiencia:** Retención de la similitud conservando únicamente el top 20% de tokens explicados.
- **Curvas de Ablación MoRF / LoRF:** Degradación en 5 bins progresivos (0%, 20%, 40%, 60%, 80%).
- **Sanity Check (Adebayo et al.):** Correlación de Spearman de las atribuciones al aleatorizar capas del Transformer en cascada.

In [ ]:
def compute_faithfulness_metrics(text_a, text_b, method_scores_a, method_scores_b, is_word_level=False):
    base_sim = compute_similarity(text_a, text_b)
    if is_word_level:
        units_a = text_a.split()
        units_b = text_b.split()
    else:
        tok_a = tokenizer.convert_ids_to_tokens(tokenizer(text_a)['input_ids'])
        tok_b = tokenizer.convert_ids_to_tokens(tokenizer(text_b)['input_ids'])
        units_a = tok_a[1:-1]
        units_b = tok_b[1:-1]
        method_scores_a = method_scores_a[1:-1]
        method_scores_b = method_scores_b[1:-1]
        
    n_a, n_b = len(units_a), len(units_b)
    k_a = max(1, int(np.ceil(0.20 * n_a)))
    k_b = max(1, int(np.ceil(0.20 * n_b)))
    
    top_idx_a = np.argsort(method_scores_a)[::-1][:k_a]
    top_idx_b = np.argsort(method_scores_b)[::-1][:k_b]
    
    # 1. Erasure (Comprehensiveness)
    erased_a = [units_a[i] for i in range(n_a) if i not in top_idx_a]
    erased_b = [units_b[i] for i in range(n_b) if i not in top_idx_b]
    erased_text_a = (' '.join(erased_a) if is_word_level else tokenizer.convert_tokens_to_string(erased_a)) or '[PAD]'
    erased_text_b = (' '.join(erased_b) if is_word_level else tokenizer.convert_tokens_to_string(erased_b)) or '[PAD]'
    sim_erased = compute_similarity(erased_text_a, erased_text_b)
    comprehensiveness = base_sim - sim_erased
    
    # 2. Sufficiency
    suff_a = [units_a[i] for i in top_idx_a]
    suff_b = [units_b[i] for i in top_idx_b]
    suff_text_a = (' '.join(suff_a) if is_word_level else tokenizer.convert_tokens_to_string(suff_a)) or '[PAD]'
    suff_text_b = (' '.join(suff_b) if is_word_level else tokenizer.convert_tokens_to_string(suff_b)) or '[PAD]'
    sim_suff = compute_similarity(suff_text_a, suff_text_b)
    sufficiency = abs(base_sim - sim_suff)
    
    return {
        "base_sim": base_sim,
        "sim_erased": sim_erased,
        "comprehensiveness": comprehensiveness,
        "sim_suff": sim_suff,
        "sufficiency": sufficiency
    }

def compute_ablation_curves(text_a, text_b, method_scores_a, method_scores_b):
    tok_a = tokenizer.convert_ids_to_tokens(tokenizer(text_a)['input_ids'])
    tok_b = tokenizer.convert_ids_to_tokens(tokenizer(text_b)['input_ids'])
    units_a, units_b = tok_a[1:-1], tok_b[1:-1]
    scores_a, scores_b = method_scores_a[1:-1], method_scores_b[1:-1]
    
    bins = [0.0, 0.2, 0.4, 0.6, 0.8]
    base_sim = compute_similarity(text_a, text_b)
    
    morf_rank_a = np.argsort(scores_a)[::-1]
    morf_rank_b = np.argsort(scores_b)[::-1]
    lorf_rank_a = np.argsort(scores_a)
    lorf_rank_b = np.argsort(scores_b)
    
    morf_sims = [base_sim]
    lorf_sims = [base_sim]
    
    for fraction in bins[1:]:
        k_a = int(np.ceil(fraction * len(units_a)))
        k_b = int(np.ceil(fraction * len(units_b)))
        
        m_mask_a = set(morf_rank_a[:k_a])
        m_mask_b = set(morf_rank_b[:k_b])
        m_tokens_a = [units_a[i] for i in range(len(units_a)) if i not in m_mask_a]
        m_tokens_b = [units_b[i] for i in range(len(units_b)) if i not in m_mask_b]
        m_text_a = tokenizer.convert_tokens_to_string(m_tokens_a) or '[PAD]'
        m_text_b = tokenizer.convert_tokens_to_string(m_tokens_b) or '[PAD]'
        morf_sims.append(compute_similarity(m_text_a, m_text_b))
        
        l_mask_a = set(lorf_rank_a[:k_a])
        l_mask_b = set(lorf_rank_b[:k_b])
        l_tokens_a = [units_a[i] for i in range(len(units_a)) if i not in l_mask_a]
        l_tokens_b = [units_b[i] for i in range(len(units_b)) if i not in l_mask_b]
        l_text_a = tokenizer.convert_tokens_to_string(l_tokens_a) or '[PAD]'
        l_text_b = tokenizer.convert_tokens_to_string(l_tokens_b) or '[PAD]'
        lorf_sims.append(compute_similarity(l_text_a, l_text_b))
        
    return {
        "bins": [0, 20, 40, 60, 80],
        "morf": morf_sims,
        "lorf": lorf_sims
    }

def run_parameter_randomization_sanity_check(sample_pair):
    text_a, text_b = sample_pair["text_a"], sample_pair["text_b"]
    orig_grad = explain_gradients(text_a, text_b, model)["ixg_a"]
    orig_ig = explain_fast_ig(text_a, text_b, steps=8, curr_model=model)["ig_a"]
    
    stages = [
        ("Original (0 Capas)", 12),
        ("Capa 11 (Top-1)", 11),
        ("Capas 11-10 (Top-2)", 10),
        ("Capas 11-8 (Top-4)", 8),
        ("Capas 11-6 (Top-6)", 6),
        ("Capas 11-0 (Todas)", 0)
    ]
    results = []
    
    def randomize_module(module):
        for m in module.modules():
            if isinstance(m, (torch.nn.Linear, torch.nn.Embedding)):
                torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None:
                    torch.nn.init.zeros_(m.bias)
            elif isinstance(m, torch.nn.LayerNorm):
                torch.nn.init.ones_(m.weight)
                torch.nn.init.zeros_(m.bias)
                
    for label, min_layer in stages:
        rand_model = copy.deepcopy(model)
        if min_layer < 12:
            for l in range(11, min_layer - 1, -1):
                randomize_module(rand_model.encoder.layer[l])
        rand_model.eval()
        
        rand_grad = explain_gradients(text_a, text_b, rand_model)["ixg_a"]
        rand_ig = explain_fast_ig(text_a, text_b, steps=8, curr_model=rand_model)["ig_a"]
        
        rho_grad, _ = spearmanr(orig_grad, rand_grad)
        rho_ig, _ = spearmanr(orig_ig, rand_ig)
        
        results.append({
            "stage": label,
            "randomized_layers_count": 12 - min_layer,
            "spearman_ixg": float(0.0 if np.isnan(rho_grad) else rho_grad),
            "spearman_ig": float(0.0 if np.isnan(rho_ig) else rho_ig)
        })
        del rand_model
        gc.collect()
        
    return results

## 8. Carga de Resultados Consolidados de la Batería Experimental

Se cargan las métricas obtenidas sobre los 10 pares curados ejecutados secuencialmente.

In [ ]:
RESULTS_FILE = os.path.join(REPORT_DIR, "xai_experimental_results.json")
with open(RESULTS_FILE, 'r', encoding='utf-8') as f:
    exp_data = json.load(f)

all_results = exp_data["pairs_data"]
latencies_summary = exp_data["latencies_ms"]
faithfulness_summary_data = exp_data["faithfulness"]
ablation_data = exp_data["ablation_curves"]
sanity_check_results = exp_data["sanity_check"]

print(f"Cargados exitosamente {len(all_results)} pares experimentales.")

## 9. Renderizado de Gráficos de Interpretabilidad y Diagnóstico

In [ ]:
# 1. Perfil de Latencia
methods = list(latencies_summary.keys())
lats = [latencies_summary[m] for m in methods]

plt.figure(figsize=(10, 5))
bars = plt.bar(methods, lats, color=['#1f77b4', '#aec7e8', '#2ca02c', '#ff7f0e', '#d62728', '#9467bd'], edgecolor='black', alpha=0.85)
plt.ylabel('Latencia Promedio (ms / par)')
plt.title('Perfil de Latencia y Eficiencia Computacional por Método XAI (Entorno CPU)')
plt.xticks(rotation=20, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
for bar, val in zip(bars, lats):
    plt.text(bar.get_x() + bar.get_width()/2.0, val + 5, f'{val:.1f} ms', ha='center', va='bottom', fontweight='bold')
plt.show()

In [ ]:
# 2. Heatmaps de Atribución Fast-IG en Pares Redundantes vs No Redundantes
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
plt.subplots_adjust(hspace=0.4, wspace=0.3)

# Par 1 Redundante
p1 = all_results[0]
toks_1a = [t.replace('##', '') for t in p1["tokens_a"][1:-1]]
toks_1b = [t.replace('##', '') for t in p1["tokens_b"][1:-1]]
sns.heatmap(np.array(p1["ig_a"][1:-1]).reshape(1, -1), annot=True, fmt=".2f", cmap="YlGnBu", xticklabels=toks_1a, yticklabels=["P1 Texto A"], ax=axes[0, 0], cbar=False)
axes[0, 0].set_title(f"Par 1 (Redundante) - Texto A (Sim: {p1['similarity']:.3f})")

sns.heatmap(np.array(p1["ig_b"][1:-1]).reshape(1, -1), annot=True, fmt=".2f", cmap="YlGnBu", xticklabels=toks_1b, yticklabels=["P1 Texto B"], ax=axes[0, 1], cbar=False)
axes[0, 1].set_title(f"Par 1 (Redundante) - Texto B")

# Par 6 No Redundante
p6 = all_results[5]
toks_6a = [t.replace('##', '') for t in p6["tokens_a"][1:-1]]
toks_6b = [t.replace('##', '') for t in p6["tokens_b"][1:-1]]
sns.heatmap(np.array(p6["ig_a"][1:-1]).reshape(1, -1), annot=True, fmt=".2f", cmap="Reds", xticklabels=toks_6a, yticklabels=["P6 Texto A"], ax=axes[1, 0], cbar=False)
axes[1, 0].set_title(f"Par 6 (No Redundante) - Texto A (Sim: {p6['similarity']:.3f})")

sns.heatmap(np.array(p6["ig_b"][1:-1]).reshape(1, -1), annot=True, fmt=".2f", cmap="Reds", xticklabels=toks_6b, yticklabels=["P6 Texto B"], ax=axes[1, 1], cbar=False)
axes[1, 1].set_title(f"Par 6 (No Redundante) - Texto B")

plt.show()

In [ ]:
# 3. Métricas de Fidelidad (Comprensividad vs. Suficiencia)
f_methods = list(faithfulness_summary_data.keys())
comp_vals = [faithfulness_summary_data[m]["mean_comprehensiveness"] for m in f_methods]
suff_vals = [faithfulness_summary_data[m]["mean_sufficiency"] for m in f_methods]

x = np.arange(len(f_methods))
width = 0.35
plt.figure(figsize=(11, 5.5))
plt.bar(x - width/2, comp_vals, width, label='Comprensividad (Erasure Top 20% - Mayor es mejor)', color='#2ca02c', edgecolor='black', alpha=0.85)
plt.bar(x + width/2, suff_vals, width, label='Suficiencia (|Δ Sim| Top 20% - Menor es mejor)', color='#1f77b4', edgecolor='black', alpha=0.85)
plt.ylabel('Δ Similitud Coseno')
plt.title('Evaluación Cuantitativa de Fidelidad por Método XAI')
plt.xticks(x, f_methods, rotation=15, ha='right')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# 4. Curvas de Ablación MoRF vs LoRF y Sanity Check
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# MoRF vs LoRF
bins = ablation_data["bins"]
axes[0].plot(bins, ablation_data["avg_morf"], 'o-', color='#d62728', linewidth=2.5, label='MoRF (Most Relevant First)')
axes[0].plot(bins, ablation_data["avg_lorf"], 's--', color='#1f77b4', linewidth=2.5, label='LoRF (Least Relevant First)')
axes[0].set_xlabel('Porcentaje de Tokens Eliminados (%)')
axes[0].set_ylabel('Similitud Coseno Promedio')
axes[0].set_title('Curvas de Ablación y Perturbación Progresiva')
axes[0].set_xticks(bins)
axes[0].set_xticklabels([f"{b}%" for b in bins])
axes[0].grid(True, linestyle='--', alpha=0.7)
axes[0].legend()

# Sanity Check
stages = [s["stage"] for s in sanity_check_results]
rho_ixg = [s["spearman_ixg"] for s in sanity_check_results]
rho_ig = [s["spearman_ig"] for s in sanity_check_results]

axes[1].plot(stages, rho_ixg, 'o-', color='#ff7f0e', linewidth=2.2, label='Input × Gradient (Spearman ρ)')
axes[1].plot(stages, rho_ig, 's-', color='#2ca02c', linewidth=2.2, label='Fast-IG (Spearman ρ)')
axes[1].axhline(0, color='gray', linestyle=':', alpha=0.7)
axes[1].set_ylabel('Correlación de Spearman (ρ) con Original')
axes[1].set_title('Prueba de Sanidad: Aleatorización en Cascada')
axes[1].set_xticklabels(stages, rotation=20, ha='right')
axes[1].grid(True, linestyle='--', alpha=0.7)
axes[1].legend()

plt.show()

## 10. Resumen Cuantitativo Consolidado y Conclusiones

In [ ]:
summary_df = pd.DataFrame([
    {
        "ID": p["id"],
        "Tipo": p["type"],
        "Similitud Coseno": f"{p['similarity']:.4f}",
        "Top Tokens A (Fast-IG)": ", ".join([p['tokens_a'][i].replace('##', '') for i in np.argsort(p['ig_a'])[::-1][:3] if p['tokens_a'][i] not in ['[CLS]', '[SEP]']][:2]),
        "Top Tokens B (Fast-IG)": ", ".join([p['tokens_b'][i].replace('##', '') for i in np.argsort(p['ig_b'])[::-1][:3] if p['tokens_b'][i] not in ['[CLS]', '[SEP]']][:2]),
    }
    for p in all_results
])
display(summary_df)

### Conclusiones y Veredicto de Interpretabilidad
1. **Fast Integrated Gradients (Fast-IG)** y **Input × Gradient** son los métodos más fieles, computacionalmente eficientes y matemáticamente rigurosos para arquitecturas siamesas con Mean-Pooling.
2. **LRP (Layer-wise Relevance Propagation)** queda formalmente descartado debido al colapso de relevancia en la capa de agregación Mean-Pooling y a la no linealidad de la métrica Coseno.
3. La prueba de sanidad de Adebayo et al. confirma que las explicaciones dependen estrictamente de los pesos entrenados del modelo.